气味与毒性的关系分析

In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')
from ucimlrepo import fetch_ucirepo

In [3]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus']=False
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 15
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['savefig.facecolor'] = 'white'

COLORS = {
    'primary': '#6F78B9',
    'edible': '#7FC6D3',
    'poisonous': '#F37252',
    'mix':'#ABE0E4',
    'light_edible': '#E3F5F6',
    'light_poisonous': '#FCB461',
}
mushroom = fetch_ucirepo(id=73) 
df = pd.concat([mushroom.data.features, mushroom.data.targets],
                   axis=1)      

In [4]:
print(f"原始数据形状: {df.shape}")
print(f"列数: {len(df.columns)}")
print(f"样本数: {len(df)}")
df

原始数据形状: (8124, 23)
列数: 23
样本数: 8124


,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,...,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat,poisonous
0,x,s,n,t,p,f,c,n,k,e,...,w,w,p,w,o,p,k,s,u,p
1,x,s,y,t,a,f,c,b,k,e,...,w,w,p,w,o,p,n,n,g,e
2,b,s,w,t,l,f,c,b,n,e,...,w,w,p,w,o,p,n,n,m,e
3,x,y,w,t,p,f,c,n,n,e,...,w,w,p,w,o,p,k,s,u,p
4,x,s,g,f,n,f,w,b,k,t,...,w,w,p,w,o,e,n,a,g,e
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8119,k,s,n,f,n,a,c,b,y,e,...,o,o,p,o,o,p,b,c,l,e
8120,x,s,n,f,n,a,c,b,y,e,...,o,o,p,n,o,p,b,v,l,e
8121,f,s,n,f,n,a,c,b,n,e,...,o,o,p,o,o,p,b,c,l,e
8122,k,y,n,f,y,f,c,n,b,t,...,w,w,p,w,o,e,w,v,l,p


处理缺失值（stalk-root列存在缺失值）

In [6]:
missing_count = (df == '?').sum().sum()
df = df.replace('?', np.nan)
print(df.isna().sum())
print(f"缺失值处理:")
print(f"  stalk-root 缺失值数量: {df['stalk-root'].isna().sum()}")
df['stalk-root'] = df['stalk-root'].fillna('unknown')

cap-shape                      0
cap-surface                    0
cap-color                      0
bruises                        0
odor                           0
gill-attachment                0
gill-spacing                   0
gill-size                      0
gill-color                     0
stalk-shape                    0
stalk-root                  2480
stalk-surface-above-ring       0
stalk-surface-below-ring       0
stalk-color-above-ring         0
stalk-color-below-ring         0
veil-type                      0
veil-color                     0
ring-number                    0
ring-type                      0
spore-print-color              0
population                     0
habitat                        0
poisonous                      0
dtype: int64
缺失值处理:
  stalk-root 缺失值数量: 2480


映射

In [8]:
ODOR_MAP = {
    'a': 'Almond',     # 杏仁味
    'l': 'Anise',      # 茴香味
    'c': 'Creosote',   # 杂酚油味
    'y': 'Fishy',      # 鱼腥味
    'f': 'Foul',       # 恶臭味
    'm': 'Musty',      # 霉味
    'n': 'None',       # 无味
    'p': 'Pungent',    # 辛辣味
    's': 'Spicy'       # 香料味
}
CLASS_MAP = {'e': '可食用',
             'p': '有毒的'}

统计

In [10]:
edible_count = len(df[df['poisonous'] == 'e'])
poisonous_count = len(df[df['poisonous'] == 'p'])
total = len(df)
print(f"可食用: {edible_count} ({edible_count/total*100:.1f}%)")
print(f"有毒: {poisonous_count} ({poisonous_count/total*100:.1f}%)")

可食用: 4208 (51.8%)
有毒: 3916 (48.2%)


In [11]:
cross_tab = pd.crosstab(df['odor'], df['poisonous'], margins=True)
print("气味-毒性交叉表 (计数):")
print(cross_tab)

cross_tab_pct = pd.crosstab(
    df['odor'], df['poisonous'], normalize='index'
) * 100
print("气味-毒性交叉表 (行百分比, %):")
print(cross_tab_pct.round(1))

print("逐气味毒性率（按有毒率升序排列）:")
odor_stats = []
for odor in sorted(df['odor'].unique()):
    subset = df[df['odor'] == odor]
    p_count = (subset['poisonous'] == 'p').sum()
    e_count = (subset['poisonous'] == 'e').sum()
    n_total = len(subset)
    p_rate = p_count / n_total * 100
    odor_stats.append({
        'odor_code': odor,
        'odor_name': ODOR_MAP[odor],
        'edible_count': e_count,
        'poisonous_count': p_count,
        'total': n_total,
        'poisonous_rate': p_rate
    })
    print(f"  {ODOR_MAP[odor]:12s} ({odor}): "
          f"总数={n_total:4d}, 可食用={e_count:4d}, 有毒={p_count:4d}, "
          f"有毒率={p_rate:5.1f}%")

odor_stats_df = pd.DataFrame(odor_stats)

气味-毒性交叉表 (计数):
poisonous     e     p   All
odor                       
a           400     0   400
c             0   192   192
f             0  2160  2160
l           400     0   400
m             0    36    36
n          3408   120  3528
p             0   256   256
s             0   576   576
y             0   576   576
All        4208  3916  8124
气味-毒性交叉表 (行百分比, %):
poisonous      e      p
odor                   
a          100.0    0.0
c            0.0  100.0
f            0.0  100.0
l          100.0    0.0
m            0.0  100.0
n           96.6    3.4
p            0.0  100.0
s            0.0  100.0
y            0.0  100.0
逐气味毒性率（按有毒率升序排列）:
  Almond       (a): 总数= 400, 可食用= 400, 有毒=   0, 有毒率=  0.0%
  Creosote     (c): 总数= 192, 可食用=   0, 有毒= 192, 有毒率=100.0%
  Foul         (f): 总数=2160, 可食用=   0, 有毒=2160, 有毒率=100.0%
  Anise        (l): 总数= 400, 可食用= 400, 有毒=   0, 有毒率=  0.0%
  Musty        (m): 总数=  36, 可食用=   0, 有毒=  36, 有毒率=100.0%
  None         (n): 总数=3528, 可食用=3408, 有毒= 120, 有毒率=

In [12]:
pure_safe = odor_stats_df[odor_stats_df['poisonous_rate'] == 0]['odor_code'].tolist()
pure_danger = odor_stats_df[odor_stats_df['poisonous_rate'] == 100]['odor_code'].tolist()
mixed = odor_stats_df[
    (odor_stats_df['poisonous_rate'] > 0) &
    (odor_stats_df['poisonous_rate'] < 100)
]['odor_code'].tolist()

print(f"--- 从数据中自动识别的气味分组 ---")
print(f"可食用 (有毒率 = 0%):    {[ODOR_MAP[o] for o in pure_safe]}")
print(f"有毒 (有毒率 = 100%):  {[ODOR_MAP[o] for o in pure_danger]}")
print(f"模糊区(0% < 有毒率 < 100%): {[ODOR_MAP[o] for o in mixed]}")
odor_stats_df['predicted_poisonous'] = odor_stats_df['poisonous_rate'].apply(
    lambda r: 'p' if r > 50 else 'e'
)
safe_odors = odor_stats_df[odor_stats_df['predicted_poisonous'] == 'e']['odor_code'].tolist()
danger_odors = odor_stats_df[odor_stats_df['predicted_poisonous'] == 'p']['odor_code'].tolist()
print(f"\n实用分类规则 (以50%为界):")
print(f"预测为edible的气味: {[ODOR_MAP[o] for o in safe_odors]}")
print(f"预测为poisonous的气味: {[ODOR_MAP[o] for o in danger_odors]}")

--- 从数据中自动识别的气味分组 ---
可食用 (有毒率 = 0%):    ['Almond', 'Anise']
有毒 (有毒率 = 100%):  ['Creosote', 'Foul', 'Musty', 'Pungent', 'Spicy', 'Fishy']
模糊区(0% < 有毒率 < 100%): ['None']

实用分类规则 (以50%为界):
预测为edible的气味: ['Almond', 'Anise', 'None']
预测为poisonous的气味: ['Creosote', 'Foul', 'Musty', 'Pungent', 'Spicy', 'Fishy']


In [13]:
print('----------预测气味辨别毒性的准确率----------')
correct_by_odor = 0
for odor in df['odor'].unique():
    subset = df[df['odor'] == odor]
    predicted = odor_stats_df.loc[
        odor_stats_df['odor_code'] == odor, 'predicted_poisonous'
    ].values[0]
    correct_by_odor += (subset['poisonous'] == predicted).sum()

accuracy = correct_by_odor / total * 100
error_cases = total - correct_by_odor
print(f"仅靠气味规则的分类准确率: {correct_by_odor}/{total} = {accuracy:.2f}%")
print(f"错误案例: {int(error_cases)}个（均来自模糊区气味 {[ODOR_MAP[o] for o in mixed]}）")

for odor in mixed:
    subset = df[df['odor'] == odor]
    predicted = odor_stats_df.loc[
        odor_stats_df['odor_code'] == odor, 'predicted_poisonous'
    ].values[0]
    wrong = (subset['poisonous'] != predicted).sum()
    print(f"  - {ODOR_MAP[odor]}: 预测为{'Edible' if predicted == 'e' else 'Poisonous'}, "
          f"错误{int(wrong)}个/共{len(subset)}个")

----------预测气味辨别毒性的准确率----------
仅靠气味规则的分类准确率: 8004/8124 = 98.52%
错误案例: 120个（均来自模糊区气味 ['None']）
  - None: 预测为Edible, 错误120个/共3528个


In [14]:
def get_odor_color(odor_code):
    """根据该气味在数据中表现出的毒性模式分配颜色"""
    if odor_code in pure_safe:
        return COLORS['edible']
    elif odor_code in pure_danger:
        return COLORS['poisonous']
    else:
        return COLORS['mix']

odor_order_by_rate = odor_stats_df.sort_values('poisonous_rate')['odor_code'].tolist()
odor_order_by_rate

['a', 'l', 'n', 'c', 'f', 'm', 'p', 's', 'y']

In [15]:
print("图1：蘑菇类别(毒性)总体分布")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors_pie = [COLORS['edible'], COLORS['poisonous']]
wedges, texts, autotexts = axes[0].pie(
    [edible_count, poisonous_count],
    labels=['Edible\n(n={:,})'.format(edible_count),
            'Poisonous\n(n={:,})'.format(poisonous_count)],
    colors=colors_pie,
    autopct='%1.1f%%',
    startangle=90,
    explode=(0, 0.05),
    textprops={'fontsize': 12, 'fontweight': 'bold'}
)
for at in autotexts:
    at.set_color('white')
    at.set_fontweight('bold')
axes[0].set_title('蘑菇食用性分布',
                  fontsize=14, fontweight='bold')

wedges2, texts2, autotexts2 = axes[1].pie(
    [edible_count, poisonous_count],
    labels=['Edible', 'Poisonous'],
    colors=colors_pie,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.78,
    wedgeprops=dict(width=0.4)
)
for at in autotexts2:
    at.set_fontweight('bold')
axes[1].set_title('饼状图：可食用与有毒对比',
                  fontsize=14, fontweight='bold')

fig.suptitle('蘑菇类别的总体分布',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig('蘑菇类别的总体分布.png',
            bbox_inches='tight', facecolor='white', edgecolor='none')
# plt.close()

图1：蘑菇类别(毒性)总体分布


In [16]:
print("图表2: 气味分布")

fig, ax = plt.subplots(figsize=(12, 7))

odor_counts = df['odor'].value_counts()
odor_labels = [ODOR_MAP[o] for o in odor_counts.index]
bar_colors = [get_odor_color(o) for o in odor_counts.index]

bars = ax.barh(range(len(odor_counts)), odor_counts.values,
               color=bar_colors, edgecolor='white',
               linewidth=0.8, height=0.7)
ax.set_yticks(range(len(odor_counts)))
ax.set_yticklabels(odor_labels, fontsize=11)
ax.set_xlabel('Number of Mushrooms', fontsize=12)
ax.set_title('图2：蘑菇气味类型的分布\n'
             '（根据发现的毒性模式进行颜色编码）',
             fontsize=14, fontweight='bold')
ax.invert_yaxis()

for i, (bar, val) in enumerate(zip(bars, odor_counts.values)):
    ax.text(val + 15, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10, fontweight='bold')

legend_elements = [
    Patch(facecolor=COLORS['edible'],
          label=f'可食用 (poison rate = 0%): '
                f'{", ".join([ODOR_MAP[o] for o in pure_safe])}'),
    Patch(facecolor=COLORS['mix'],
          label=f'模糊区 (0% < rate < 100%): '
                f'{", ".join([ODOR_MAP[o] for o in mixed])}'),
    Patch(facecolor=COLORS['poisonous'],
          label=f'有毒 (poison rate = 100%): '
                f'{", ".join([ODOR_MAP[o] for o in pure_danger])}'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8, framealpha=0.9)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig('蘑菇气味类型的分布.png',
            bbox_inches='tight', facecolor='white', edgecolor='none')
plt.close()

图表2: 气味分布


In [17]:
print("图表3: 气味与毒性堆叠柱状图")

fig, ax = plt.subplots(figsize=(12, 7))

plot_df = odor_stats_df.sort_values('poisonous_rate')

x_labels = plot_df['odor_name'].tolist()
x = range(len(x_labels))
width = 0.65

bars_e = ax.bar(x, plot_df['edible_count'], width,
                label='Edible', color=COLORS['edible'],
                edgecolor='white', linewidth=0.5)
bars_p = ax.bar(x, plot_df['poisonous_count'], width,
                bottom=plot_df['edible_count'],
                label='Poisonous', color=COLORS['poisonous'],
                edgecolor='white', linewidth=0.5)

for i, row in enumerate(plot_df.itertuples()):
    y_pos = row.edible_count + row.poisonous_count + 20
    color = COLORS['poisonous'] if row.poisonous_rate > 50 else COLORS['edible']
    ax.text(i, y_pos, f'{row.poisonous_rate:.1f}%',
            ha='center', fontsize=9, fontweight='bold', color=color)

ax.set_xticks(x)
ax.set_xticklabels(x_labels, fontsize=10, rotation=30, ha='right')
ax.set_ylabel('Number of Mushrooms', fontsize=12)
ax.set_title('图3：按气味类型划分的蘑菇毒性（堆积条）\n'
'（按发现的中毒率排序——从低到高）',
             fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=11)
ax.set_ylim(0, max(plot_df['total']) * 1.15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig('按气味类型划分的蘑菇毒性.png',
            bbox_inches='tight', facecolor='white', edgecolor='none')
plt.close()

图表3: 气味与毒性堆叠柱状图


In [18]:
print("图表4: 各气味毒性率排序")

fig, ax = plt.subplots(figsize=(12, 7))

rates_sorted = odor_stats_df.sort_values('poisonous_rate')
bar_colors_rate = [get_odor_color(o) for o in rates_sorted['odor_code']]

bars = ax.barh(range(len(rates_sorted)),
               rates_sorted['poisonous_rate'],
               color=bar_colors_rate,
               edgecolor='white', linewidth=0.8, height=0.65)

ax.set_yticks(range(len(rates_sorted)))
ax.set_yticklabels(rates_sorted['odor_name'].tolist(), fontsize=11)
ax.invert_yaxis()
ax.set_xlabel('Poisonous Rate (%)', fontsize=12)
ax.set_title('图4：按气味类型分类的中毒率'
'（发现的分组：0%=可食用，100%=有毒）',
             fontsize=14, fontweight='bold')

ax.axvline(x=50, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.text(51, len(rates_sorted) - 0.5, '50% Threshold',
        fontsize=9, color='gray', alpha=0.8)

for i, row in enumerate(rates_sorted.itertuples()):
    ax.text(row.poisonous_rate + 1,
            bars[i].get_y() + bars[i].get_height() / 2,
            f'{row.poisonous_rate:.1f}% (n={int(row.total)})',
            va='center', fontsize=9, fontweight='bold')

ax.set_xlim(0, 110)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig('按气味类型分类的中毒率.png',
            bbox_inches='tight', facecolor='white', edgecolor='none')
plt.close()

图表4: 各气味毒性率排序


In [38]:
print("图表5: 气味-毒性热力图")

fig, ax = plt.subplots(figsize=(8, 5))

heatmap_data = pd.crosstab(
    df['odor'].map(ODOR_MAP),
    df['poisonous'].map(CLASS_MAP)
)

heatmap_data = heatmap_data.loc[
    heatmap_data.sum(axis=1).sort_values(ascending=False).index
]

sns.heatmap(heatmap_data,
            annot=True, fmt='d', cmap='RdBu_r',
            linewidths=1, linecolor='white',
            cbar_kws={'label': 'Count'},
            ax=ax,
            center=heatmap_data.values.mean(),
            xticklabels=True, yticklabels=True,
            annot_kws={'fontsize': 11, 'fontweight': 'bold'})

ax.set_title('图5：气味与毒性热图'
'（发现：2种安全气味，6种危险气味，1种混合气味）',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Mushroom Class', fontsize=12)
ax.set_ylabel('Odor Type', fontsize=12)
plt.tight_layout()
fig.savefig('气味与毒性热图.png',
            bbox_inches='tight', facecolor='white', edgecolor='none')
plt.close()

图表5: 气味-毒性热力图


In [42]:
print("图表6: 气味-毒性百分比对比")

fig, ax = plt.subplots(figsize=(12, 7))

pct_df = odor_stats_df.sort_values('poisonous_rate').copy()
pct_df['edible_pct'] = 100 - pct_df['poisonous_rate']

x = range(len(pct_df))
width = 0.35

ax.bar([i - width/2 for i in x], pct_df['edible_pct'], width,
       label='Edible (%)', color=COLORS['edible'],
       edgecolor='white', linewidth=0.5)

ax.bar([i + width/2 for i in x], pct_df['poisonous_rate'], width,
       label='Poisonous (%)', color=COLORS['poisonous'],
       edgecolor='white', linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(pct_df['odor_name'].tolist(), fontsize=10,
                   rotation=30, ha='right')
ax.set_ylabel('Percentage (%)', fontsize=12)
ax.set_title('图6:按气味划分的可食用与有毒百分比'
'（极端双峰模式：大多数气味要么100%安全，要么100%有毒）',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, 110)
ax.axhline(y=50, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig('按气味划分的可食用与有毒百分比.png',
            bbox_inches='tight', facecolor='white', edgecolor='none')
plt.close()

图表6: 气味-毒性百分比对比


In [50]:
print("图表7: 综合仪表盘")

fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.3)

# ---- (0,0) 类别分布饼图 ----
ax1 = fig.add_subplot(gs[0, 0])
ax1.pie([edible_count, poisonous_count],
        labels=['Edible\n({:,})'.format(edible_count),
                'Poisonous\n({:,})'.format(poisonous_count)],
        colors=[COLORS['edible'], COLORS['poisonous']],
        autopct='%1.1f%%', startangle=90, explode=(0, 0.05),
        textprops={'fontsize': 10})
ax1.set_title('Class Distribution', fontsize=13, fontweight='bold')

# ---- (0,1) 气味分布柱状图 ----
ax2 = fig.add_subplot(gs[0, 1])
odor_counts_plot = df['odor'].value_counts()
odor_labels_plot = [ODOR_MAP[o] for o in odor_counts_plot.index]
c2 = [get_odor_color(o) for o in odor_counts_plot.index]
ax2.bar(range(len(odor_counts_plot)), odor_counts_plot.values,
        color=c2, edgecolor='white')
ax2.set_xticks(range(len(odor_counts_plot)))
ax2.set_xticklabels(odor_labels_plot, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('Count', fontsize=10)
ax2.set_title('Odor Distribution\n(Colored by discovered toxicity)',
              fontsize=13, fontweight='bold')

# ---- (0,2) 毒性率排序 ----
ax3 = fig.add_subplot(gs[0, 2])
c3 = [get_odor_color(o) for o in rates_sorted['odor_code']]
ax3.barh(range(len(rates_sorted)), rates_sorted['poisonous_rate'],
         color=c3, edgecolor='white', height=0.7)
ax3.set_yticks(range(len(rates_sorted)))
ax3.set_yticklabels(rates_sorted['odor_name'].tolist(), fontsize=9)
ax3.set_xlabel('Poisonous Rate (%)', fontsize=10)
ax3.set_title('Toxicity Rate by Odor', fontsize=13, fontweight='bold')
ax3.axvline(x=50, color='gray', linestyle='--', linewidth=0.8)

# ---- (1,0) 堆叠柱状图 ----
ax4 = fig.add_subplot(gs[1, 0])
ax4.bar(x, pct_df['edible_count'], width, label='Edible',
        color=COLORS['edible'])
ax4.bar(x, pct_df['poisonous_count'], width,
        bottom=pct_df['edible_count'], label='Poisonous',
        color=COLORS['poisonous'])
ax4.set_xticks(x)
ax4.set_xticklabels(pct_df['odor_name'].tolist(),
                    rotation=45, ha='right', fontsize=8)
ax4.set_ylabel('Count', fontsize=10)
ax4.set_title('Odor vs Toxicity (Stacked)', fontsize=13, fontweight='bold')
ax4.legend(fontsize=9, loc='upper right')

# ---- (1,1) 热力图 ----
ax5 = fig.add_subplot(gs[1, 1])
sns.heatmap(heatmap_data, annot=True, fmt='d', cmap='RdBu_r',
            linewidths=1, linecolor='white',
            cbar_kws={'label': 'Count'},
            ax=ax5, xticklabels=True, yticklabels=True,
            annot_kws={'fontsize': 9})
ax5.set_title('Heatmap: Odor × Class', fontsize=13, fontweight='bold')

# ---- (1,2) 百分比分组对比 ----
ax6 = fig.add_subplot(gs[1, 2])
w2 = 0.35
ax6.bar([i - w2/2 for i in x], pct_df['edible_pct'], w2,
        label='Edible', color=COLORS['edible'])
ax6.bar([i + w2/2 for i in x], pct_df['poisonous_rate'], w2,
        label='Poisonous', color=COLORS['poisonous'])
ax6.set_xticks(x)
ax6.set_xticklabels(pct_df['odor_name'].tolist(),
                    rotation=45, ha='right', fontsize=8)
ax6.set_ylabel('Percentage (%)', fontsize=10)
ax6.set_title('Percentage by Odor', fontsize=13, fontweight='bold')
ax6.legend(fontsize=9)
ax6.axhline(y=50, color='gray', linestyle='--', linewidth=0.8)

fig.suptitle('蘑菇气味与毒性——综合分析仪表板'
'（从数据中发现的所有分组，不是预定义的）',
             fontsize=17, fontweight='bold', y=1.01)
fig.savefig('蘑菇气味与毒性——综合分析仪表板.png',
            bbox_inches='tight', facecolor='white', edgecolor='none')
plt.close()

图表7: 综合仪表盘
